# 编排器-工作者工作流

## 简介

你是否需要在同一任务上获得多个视角，但无法提前预测哪些视角最有价值？编排器-工作者模式通过让一个中央 LLM 分析每个独特任务并动态确定最佳子任务来委派给专业的工作者 LLM 来解决这个问题。

传统方法要么需要多次手动提示，要么使用硬编码的并行化，无论上下文如何都生成相同的变体。

通过这种方法，编排器 LLM 分析任务，确定针对此特定案例最有价值的变体，然后委派给生成每个变体的工作者 LLM。

### 你将构建

一个接收产品描述请求并执行以下操作的系统：

1. 分析哪些类型的营销文案有价值
2. 动态生成工作者的专业任务描述
3. 生成针对不同受众优化的多个内容变体
4. 返回所有工作者的协调结果

### 前置要求

- Python 3.9 或更高版本
- 设置 Anthropic API 密钥为环境变量：`export ANTHROPIC_API_KEY='your-key'`
- 对提示工程的基本了解
- 熟悉 Python 类和类型提示


### 何时使用此工作流

此工作流非常适合无法提前预测所需子任务的复杂任务。与简单并行化的主要区别在于其灵活性——子任务不是预定义的，而是由编排器根据特定输入确定的。

**在此情况下使用此模式：**

- 任务需要多种不同的方法或视角
- 最佳子任务取决于特定输入
- 你需要比较不同的策略或风格

**不要在此情况下使用此模式：**

- 你有简单的单输出任务（不必要的复杂性）
- 延迟至关重要（多次 LLM 调用会增加开销）
- 子任务可预测且可以预定义（使用更简单的并行化）

## 工作原理

编排器-工作者模式分两个阶段运行：

1. **分析与规划阶段**：编排器 LLM 接收任务和上下文，分析哪些方法有价值，并以 XML 格式生成结构化的子任务描述。

2. **执行阶段**：每个工作者 LLM 接收：
   - 原始任务以提供上下文
   - 其特定的子任务类型和描述
   - 提供的任何额外上下文

编排器在*运行时*决定创建哪些子任务，这使其比预定义的并行工作流更具适应性。

## 设置

### 安装
```bash
pip install anthropic
```

### 辅助函数
此示例使用 `util.py` 中的辅助函数来进行 LLM 调用和解析 XML 响应：

- `llm_call(prompt, system_prompt="", model="claude-sonnet-4-6")`：向 Claude 发送提示并返回文本响应
- `extract_xml(text, tag)`：使用正则表达式从 XML 标签中提取内容

这些工具处理 API 身份验证（从环境读取 `ANTHROPIC_API_KEY`）并为编排器-工作者模式提供简单的接口。你可以在 [util.py](util.py) 中查看完整实现。

## 实现

`FlexibleOrchestrator` 类协调两阶段工作流：

**关键设计决策：**
- 提示是接受运行时变量（`task`、`context`）的模板以提供灵活性
- XML 用于结构化输出解析（可靠且对语言模型友好的格式）
- 工作者接收原始任务及其特定指令以获得更好的上下文
- 错误处理验证工作者返回非空响应

实现包括：
- `parse_tasks()`：将编排器的 XML 输出解析为结构化任务字典
- `FlexibleOrchestrator.process()`：调用编排器然后调用工作者的主要协调逻辑
- 响应验证以捕获和处理空的工作者输出

In [ ]:
from util import extract_xml, llm_call

# Model configuration
MODEL = "claude-sonnet-4-6"  # Fast, capable model for both orchestrator and workers


def parse_tasks(tasks_xml: str) -> list[dict]:
    """Parse XML tasks into a list of task dictionaries."""
    tasks = []
    current_task = {}

    for line in tasks_xml.split("\n"):
        line = line.strip()
        if not line:
            continue

        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)

    return tasks


class FlexibleOrchestrator:
    """Break down tasks and run them in parallel using worker LLMs."""

    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
        model: str = MODEL,
    ):
        """Initialize with prompt templates and model selection."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt
        self.model = model

    def _format_prompt(self, template: str, **kwargs) -> str:
        """Format a prompt template with variables."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}") from e

    def process(self, task: str, context: dict | None = None) -> dict:
        """Process task by breaking it down and running subtasks in parallel."""
        context = context or {}

        # Step 1: Get orchestrator response
        orchestrator_input = self._format_prompt(self.orchestrator_prompt, task=task, **context)
        orchestrator_response = llm_call(orchestrator_input, model=self.model)

        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)

        print("\n" + "=" * 80)
        print("ORCHESTRATOR ANALYSIS")
        print("=" * 80)
        print(f"\n{analysis}\n")

        print("\n" + "=" * 80)
        print(f"IDENTIFIED {len(tasks)} APPROACHES")
        print("=" * 80)
        for i, task_info in enumerate(tasks, 1):
            print(f"\n{i}. {task_info['type'].upper()}")
            print(f"   {task_info['description']}")

        print("\n" + "=" * 80)
        print("GENERATING CONTENT")
        print("=" * 80 + "\n")

        # Step 2: Process each task
        worker_results = []
        for i, task_info in enumerate(tasks, 1):
            print(f"[{i}/{len(tasks)}] Processing: {task_info['type']}...")

            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_type=task_info["type"],
                task_description=task_info["description"],
                **context,
            )

            worker_response = llm_call(worker_input, model=self.model)
            worker_content = extract_xml(worker_response, "response")

            # Validate worker response - handle empty outputs
            if not worker_content or not worker_content.strip():
                print(f"⚠️  Warning: Worker '{task_info['type']}' returned no content")
                worker_content = f"[Error: Worker '{task_info['type']}' failed to generate content]"

            worker_results.append(
                {
                    "type": task_info["type"],
                    "description": task_info["description"],
                    "result": worker_content,
                }
            )

        # Display results
        print("\n" + "=" * 80)
        print("RESULTS")
        print("=" * 80)
        for i, result in enumerate(worker_results, 1):
            print(f"\n{'-' * 80}")
            print(f"Approach {i}: {result['type'].upper()}")
            print(f"{'-' * 80}")
            print(f"\n{result['result']}\n")

        return {
            "analysis": analysis,
            "worker_results": worker_results,
        }

## 示例用例：营销变体生成

现在让我们通过一个实际示例来看看编排器-工作者模式的实际应用：为产品生成多种风格的营销文案。

**为什么此示例很好地展示了该模式：**
- 不同的产品受益于不同的营销角度
- "最佳"变体取决于特定的产品功能和目标受众
- 编排器可以根据输入调整其策略，而不是使用固定模板

**提示设计说明：**
- 编排器提示要求 2-3 种方法并提供 XML 结构指导
- 工作者提示为工作者提供完整上下文（原始任务、其风格和指导原则）
- 两个提示都使用清晰的 XML 格式以确保可靠的解析

In [9]:
ORCHESTRATOR_PROMPT = """
Analyze this task and break it down into 2-3 distinct approaches:

Task: {task}

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
    <type>formal</type>
    <description>Write a precise, technical version that emphasizes specifications</description>
    </task>
    <task>
    <type>conversational</type>
    <description>Write an engaging, friendly version that connects with readers</description>
    </task>
</tasks>
"""

WORKER_PROMPT = """
Generate content based on:
Task: {original_task}
Style: {task_type}
Guidelines: {task_description}

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"""


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task="Write a product description for a new eco-friendly water bottle",
    context={
        "target_audience": "environmentally conscious millennials",
        "key_features": ["plastic-free", "insulated", "lifetime warranty"],
    },
)


ORCHESTRATOR ANALYSIS


This task requires creating marketing copy for an eco-friendly water bottle. The core challenge is balancing product information with persuasive messaging while highlighting the environmental benefits. Different approaches would serve distinct marketing channels and audience segments:

1. A **feature-focused technical approach** would appeal to detail-oriented consumers who make purchasing decisions based on specifications, materials, and measurable environmental impact. This serves e-commerce listings and comparison shopping.

2. A **lifestyle-oriented emotional approach** would connect with values-driven consumers through storytelling and aspirational messaging, emphasizing how the product fits into an eco-conscious lifestyle. This serves social media and brand-building content.

3. A **benefit-driven practical approach** would focus on solving everyday problems while weaving in sustainability advantages, appealing to mainstream consumers who want both functi

## 总结

你现在已经实现了一个编排器-工作者模式，该模式根据特定输入动态调整其任务分解。此模式生成了多个营销文案变体——每个变体都针对不同的受众和上下文量身定制——无需你预先定义这些变体应该是什么。

### 关键要点

**模式优势：**
- **适应性**：编排器为每个独特输入确定最佳方法
- **灵活性**：通过更改提示轻松应用于不同领域
- **结构化协调**：基于 XML 的通信确保可靠的解析
- **错误恢复**：验证捕获并处理工作者故障

**此模式的优势场景：**
- 需要多个视角的内容生成（营销、文档、创意写作）
- 受益于不同分析视角的分析任务
- 分解策略取决于问题的问题解决

### 局限性和注意事项

**成本和延迟：**
- 需要 N+1 次 LLM 调用（1 个编排器 + N 个工作者）
- 此实现中的顺序处理（工作者一次运行一个）
- 为了获得更好的性能，考虑使用 `asyncio` 或线程池并行化工作者调用

**何时不使用此模式：**
- 具有单一清晰输出的简单任务（增加的复杂性不合理）
- 延迟关键型应用（多次 API 调用增加开销）
- 子任务始终相同的任务（改为使用预定义的并行化）

**需考虑的故障模式：**
- 编排器可能无法最优地分解任务（提示工程至关重要）
- 工作者可能返回空或格式错误的响应（我们通过验证处理此问题）
- 如果模型不完全遵循格式，XML 解析可能会失败（考虑使用 JSON 作为替代）

### 后续步骤

**增强此实现：**
1. 使用 `asyncio` 添加并行工作者执行以获得更好的性能
2. 为失败的工作者实现重试逻辑
3. 添加一个合成阶段，其中 LLM 组合工作者输出
4. 尝试不同的编排器策略（例如，要求更多/更少的子任务）

**适应你的用例：**
- 修改编排器提示以指导你领域的任务分解
- 调整工作者提示以提供特定领域的指令
- 添加与你的应用相关的上下文参数
- 考虑对编排器使用 Claude Opus，对工作者使用 Claude Haiku，以优化成本与质量